# 05 — Methodological Critic Audit & 4-Pillar Report Synthesis
**AutoDS Scientific Methodology Demonstration**

This notebook demonstrates the Critic leakage and overfitting audit, and final report compilation with structured 4-pillar evidence.


In [1]:
import pandas as pd
from backend.app.tools.preprocessor import prepare_train_test_split
from backend.app.tools.ml_trainer import train_and_evaluate_model, evaluate_locked_champion_on_holdout
from backend.app.tools.critic import critique_experiment
from backend.app.tools.reporter import generate_full_markdown_report
from backend.app.tools.data_profiler import profile_dataset

df = pd.read_csv("data/raw/31513e04_winequality-red.csv", sep=None, engine="python")
profile = profile_dataset(df)
X_train, X_test, y_train, y_test, prep = prepare_train_test_split(df, target_column="quality", problem_type="classification", test_size=0.2, random_state=42)
exp = train_and_evaluate_model("LogisticRegression", "classification", X_train, y_train, feature_names=prep.feature_names, cv_folds=3, track_mlflow=False)
evaluate_locked_champion_on_holdout(exp, X_train, y_train, X_test, y_test, track_mlflow=False)
print("Pipeline evaluated. Proceeding to Critic Audit.")


Pipeline evaluated. Proceeding to Critic Audit.



## 1. Methodological Critic Audit
The Critic evaluates train/test accuracy divergence, target leakage indicators, and feature collinearity.


In [2]:
critic_report = critique_experiment(
    model_name=exp["model_name"],
    problem_type="classification",
    metrics=exp["metrics"],
    feature_names=prep.feature_names,
    validation_strategy="Stratified 3-Fold Cross-Validation",
    target_column="quality"
)

print(f"Critic Audit Status: {critic_report['audit_status']}")
print(f"Total Critic Findings: {len(critic_report['findings'])}")
for f in critic_report['findings']:
    print(f"  [{f['severity'].upper()}] {f['issue_type']}: {f['description']}")


Critic Audit Status: PASSED
Total Critic Findings: 0



## 2. 4-Pillar Evidence Report Compilation
The final markdown report synthesizes Observed Facts, Model Evidence, Actionable Recommendations, and Causal Limitations.


In [3]:
insights = [
    {
        "category": "observed_facts",
        "title": "Empirical Sample Volume",
        "finding": f"Audited {df.shape[0]:,} red wine samples across {df.shape[1]-1} chemical features.",
        "evidence": f"Total samples: {df.shape[0]:,}",
        "confidence": "High"
    },
    {
        "category": "model_derived",
        "title": "Predictive Discriminability",
        "finding": f"Champion {exp['model_name']} achieved Macro-AUC of {exp['metrics']['test']['roc_auc']:.4f}.",
        "evidence": f"Holdout Macro-AUC: {exp['metrics']['test']['roc_auc']:.4f}",
        "confidence": "High"
    }
]

report_md = generate_full_markdown_report(
    dataset_name="winequality-red.csv",
    user_goal="Predict wine quality ratings based on physicochemical properties",
    problem_type="classification",
    target_column="quality",
    validation_strategy="Stratified 3-Fold Cross-Validation",
    profile_summary=profile,
    experiment_results=[exp],
    best_experiment=exp,
    critic_audit=critic_report,
    business_insights=insights,
    artifact_paths=["reports/artifacts/demo_roc.png"]
)

print(f"Generated Audit-Ready Markdown Report ({len(report_md):,} characters):\n")
print(report_md)


Generated Audit-Ready Markdown Report (5,705 characters):

# AutoDS Autonomous Data Science Report
**Dataset:** `winequality-red.csv`  
**Generated:** 2026-08-18 17:42:45 UTC  
**Status:** Verified & Evidence-Backed  

---

## 1. Executive Summary
**Objective:** Predict wine quality ratings based on physicochemical properties

AutoDS autonomously profiled the dataset, identified a **CLASSIFICATION** task targeting `quality` using **Stratified 3-Fold Cross-Validation** validation. 
The champion model selected is **LogisticRegression**, achieving a Test **ROC-AUC of 0.7821**, **PR-AUC of 0.3724**, **Balanced Accuracy of 0.2673**, and **Positive-Class F1 of 0.2676** (F2: 0.2676).

## 2. Dataset Overview & Data Quality Profile (Observed Facts)
- **Dimensions:** 1,599 rows × 12 columns
- **Total Missing Cells:** 0.0%
- **Duplicate Rows Removed:** 240
- **Target Column:** `quality`
- **Validation Strategy:** `Stratified 3-Fold Cross-Validation`


## 3. Model Leaderboard & Multi-Metric Evalua